# Day 03: PyTorch Basics — Tensors, Autograd, and GPU

**Goal:** Learn PyTorch — the tool that automates everything we did by hand in Day 02.

### The Problem with Day 02

In Day 02, we computed gradients by hand:
```python
dw = np.mean(2 * (y_pred - y_true) * X)  # we derived this formula manually
db = np.mean(2 * (y_pred - y_true))
```

This worked for 2 parameters. But a real LLM has **billions** of parameters. You can't hand-derive billions of gradient formulas.

### The Solution: PyTorch

PyTorch does two things that make deep learning possible:
1. **Autograd** — automatically computes gradients for ANY computation. You write the forward pass, PyTorch figures out the backward pass.
2. **GPU support** — runs matrix math 10-100x faster on a GPU.

Let's see it in action.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available (Apple GPU): {torch.backends.mps.is_available()}")

## 1. Tensors — NumPy Arrays with Superpowers

A PyTorch **tensor** is basically a NumPy array that can:
- Run on GPU
- Automatically track gradients

Everything you learned about shapes, reshaping, and operations in Day 01 applies here. The API is almost identical.

In [ ]:
# Creating tensors — looks just like NumPy!

# From a Python list
a = torch.tensor([1.0, 2.0, 3.0])
print(f"From list:   {a}, shape: {a.shape}, dtype: {a.dtype}")

# Common initializations
zeros = torch.zeros(2, 3)       # 2×3 matrix of zeros
ones = torch.ones(2, 3)         # 2×3 matrix of ones
random = torch.randn(2, 3)      # 2×3 random from normal distribution

print(f"\nZeros:\n{zeros}")
print(f"\nOnes:\n{ones}")
print(f"\nRandom (normal distribution):\n{random}")

# arange and linspace — same as NumPy
r = torch.arange(0, 10, 2)      # [0, 2, 4, 6, 8]
l = torch.linspace(0, 1, 5)     # [0.0, 0.25, 0.5, 0.75, 1.0]
print(f"\narange: {r}")
print(f"linspace: {l}")

In [ ]:
# Operations — identical to NumPy

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("Element-wise operations (same as NumPy):")
print(f"  a + b = {a + b}")
print(f"  a * b = {a * b}")       # element-wise multiply
print(f"  a ** 2 = {a ** 2}")     # element-wise square

print(f"\nDot product:")
print(f"  a @ b = {a @ b}")       # 1*4 + 2*5 + 3*6 = 32

# Matrix multiplication
A = torch.randn(2, 3)
B = torch.randn(3, 4)
C = A @ B
print(f"\nMatrix multiply: {A.shape} @ {B.shape} = {C.shape}")

# Aggregations
data = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
print(f"\nsum: {data.sum()}, mean: {data.mean()}, max: {data.max()}")

### NumPy ↔ PyTorch conversion

You'll often need to switch between the two. NumPy is used for plotting (matplotlib) and data prep. PyTorch is used for training.

In [ ]:
# NumPy → PyTorch
np_array = np.array([1.0, 2.0, 3.0])
tensor = torch.from_numpy(np_array)
print(f"NumPy → Tensor: {tensor}")

# PyTorch → NumPy
back_to_numpy = tensor.numpy()
print(f"Tensor → NumPy: {back_to_numpy}")

# WARNING: they share memory! Changing one changes the other.
np_array[0] = 999
print(f"\nChanged NumPy[0] to 999")
print(f"Tensor is now: {tensor}")  # also changed!
print("⚠️  They share memory — use .clone() if you need a separate copy")

# Safe copy
safe_tensor = torch.from_numpy(np_array).clone()  # independent copy

## 2. Autograd — The Magic of Automatic Gradients

This is the big one. In Day 02, we derived gradient formulas by hand. PyTorch does this **automatically**.

### How it works:

1. Create a tensor with `requires_grad=True` — this tells PyTorch "I want to compute gradients for this"
2. Do any math you want — PyTorch secretly records every operation
3. Call `.backward()` on the result — PyTorch walks backwards through the recorded operations (chain rule!) and computes all gradients
4. Read the gradient from `.grad`

Let's start with the simplest possible example:

In [ ]:
# Simplest autograd example: f(x) = x²,  derivative = 2x

# Step 1: create tensor with requires_grad=True
x = torch.tensor(3.0, requires_grad=True)

# Step 2: do some computation
y = x ** 2    # y = 9.0

# Step 3: compute gradients automatically
y.backward()

# Step 4: read the gradient
print(f"x = {x.item()}")
print(f"y = x² = {y.item()}")
print(f"dy/dx (from PyTorch) = {x.grad.item()}")
print(f"dy/dx (we know it's 2x = 2×3) = {2 * 3.0}")
print(f"\nPyTorch computed the derivative automatically! ✓")

### What just happened step by step:

```
x = 3.0  (with requires_grad=True — PyTorch is now watching)
    ↓
y = x²   (PyTorch records: "y came from squaring x")
    ↓
y.backward()  (PyTorch thinks: "derivative of x² is 2x = 2×3 = 6")
    ↓
x.grad = 6.0  (the answer is stored here)
```

**The key:** We never wrote `2 * x` anywhere. PyTorch figured out the derivative formula on its own.

Now let's try something more complex — a chain of operations (which would need the chain rule by hand):

In [ ]:
# More complex: y = (3x + 2)²  — this needed the chain rule in Day 02!

x = torch.tensor(1.0, requires_grad=True)

# Chain of operations (PyTorch records each step)
a = 3 * x + 2    # step 1: a = 3x + 2 = 5
y = a ** 2        # step 2: y = a² = 25

y.backward()

print(f"y = (3x + 2)² at x = {x.item()}")
print(f"y = {y.item()}")
print(f"dy/dx (PyTorch autograd): {x.grad.item()}")
print(f"dy/dx (chain rule by hand: 6(3x+2) = 6×5): {6 * (3 * 1.0 + 2)}")
print(f"\nPyTorch applied the chain rule automatically! ✓")

In [ ]:
# Multiple variables — gradient for EACH one

w = torch.tensor(2.0, requires_grad=True)   # weight
b = torch.tensor(1.0, requires_grad=True)   # bias
x = torch.tensor(3.0)                        # input (no grad needed — it's data, not a weight)

# Forward pass: a simple neuron
y = w * x + b       # y = 2*3 + 1 = 7
loss = (y - 10) ** 2  # loss = (7 - 10)² = 9   (target is 10)

loss.backward()

print(f"y = w*x + b = {w.item()}*{x.item()} + {b.item()} = {y.item()}")
print(f"loss = (y - 10)² = ({y.item()} - 10)² = {loss.item()}")
print(f"")
print(f"∂loss/∂w (PyTorch): {w.grad.item()}")
print(f"∂loss/∂b (PyTorch): {b.grad.item()}")
print(f"")
print(f"What these mean:")
print(f"  w.grad = {w.grad.item()} → if we increase w by a tiny bit, loss changes by {w.grad.item()}")
print(f"  b.grad = {b.grad.item()} → if we increase b by a tiny bit, loss changes by {b.grad.item()}")
print(f"")
print(f"Both are negative → increasing w or b will DECREASE the loss → good!")
print(f"This tells the optimizer: make w and b bigger to get closer to target.")

### Notice what we DIDN'T do:
- We didn't write any gradient formula
- We didn't use the chain rule by hand
- We just wrote the forward computation (`y = w*x + b`, `loss = (y-10)²`) and called `.backward()`

PyTorch figured out both `∂loss/∂w` and `∂loss/∂b` automatically. **This scales to billions of parameters.**

---

## 3. Gradient Descent in PyTorch — Day 02 Rewritten

Let's redo our Day 02 linear regression, but this time PyTorch computes the gradients for us.

### Important gotcha: `w.grad.zero_()`

PyTorch **accumulates** gradients by default. Each time you call `.backward()`, it ADDS the new gradient to whatever was already in `.grad`. This is occasionally useful, but usually you want to clear them first.

```python
# Without zeroing: grad after step 1 = 5, after step 2 = 5+3=8 (WRONG!)
# With zeroing:    grad after step 1 = 5, after step 2 = 3 (CORRECT!)
```

Forgetting to zero gradients is one of the most common PyTorch bugs.

In [ ]:
# Day 02 linear regression — now with PyTorch autograd!
# Goal: learn y = 3x + 7 from noisy data

# Generate data (same as Day 02)
torch.manual_seed(42)
X = torch.rand(50) * 10 - 5       # 50 points between -5 and 5
y_true = 3 * X + 7 + torch.randn(50) * 2  # y = 3x + 7 + noise

# Initialize learnable parameters
w = torch.tensor(0.0, requires_grad=True)  # should learn → 3
b = torch.tensor(0.0, requires_grad=True)  # should learn → 7
learning_rate = 0.01

losses = []

print(f"Starting: w={w.item():.3f} (target: 3.0), b={b.item():.3f} (target: 7.0)\n")

for epoch in range(100):
    # === FORWARD PASS ===
    y_pred = w * X + b
    
    # === COMPUTE LOSS ===
    loss = torch.mean((y_pred - y_true) ** 2)
    losses.append(loss.item())
    
    # === BACKWARD PASS (this is where the magic happens!) ===
    loss.backward()  # PyTorch computes dw and db automatically
    
    # === UPDATE WEIGHTS ===
    with torch.no_grad():  # don't track gradients for the update itself
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    
    # === ZERO GRADIENTS (crucial!) ===
    w.grad.zero_()
    b.grad.zero_()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}: loss={loss.item():.3f}, w={w.item():.3f}, b={b.item():.3f}")

print(f"\nFinal:  w={w.item():.3f} (target: 3.0), b={b.item():.3f} (target: 7.0)")
print(f"\nCompare with Day 02: SAME result, but we never wrote a gradient formula!")

### Side by side: Day 02 vs Day 03

```python
# DAY 02 — Manual gradients               # DAY 03 — PyTorch autograd
# ─────────────────────────                # ──────────────────────────
y_pred = w * X + b                         y_pred = w * X + b              # same!
loss = np.mean((y_pred - y_true) ** 2)     loss = torch.mean((...) ** 2)   # same!
                                           
dw = np.mean(2*(y_pred-y_true)*X)  # HAND  loss.backward()                 # AUTOMATIC!
db = np.mean(2*(y_pred-y_true))    # HAND  
                                           
w = w - lr * dw                            w -= lr * w.grad                # same idea
b = b - lr * db                            b -= lr * b.grad
                                           w.grad.zero_()                  # new: must zero
                                           b.grad.zero_()
```

The forward pass is identical. The only difference: we replaced 2 hand-derived gradient lines with `loss.backward()`.

For 2 parameters, the hand-derived version is fine. For 175 billion parameters, you NEED autograd.

---

## 4. Using an Optimizer — The Standard PyTorch Way

Manually writing `w -= lr * w.grad` works, but PyTorch has built-in optimizers that handle the update (and can do smarter things like momentum and adaptive learning rates).

The standard training loop is just 3 lines inside the loop:
```python
optimizer.zero_grad()  # clear old gradients
loss.backward()         # compute new gradients  
optimizer.step()        # update all weights
```

This 3-line pattern is used in every PyTorch project, from tutorials to production LLMs.

In [ ]:
# Same linear regression, but using PyTorch optimizer

# Fresh data
torch.manual_seed(42)
X = torch.rand(50) * 10 - 5
y_true = 3 * X + 7 + torch.randn(50) * 2

# Learnable parameters
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

# SGD = Stochastic Gradient Descent (the basic optimizer)
# It does exactly: param -= lr * param.grad for each parameter
optimizer = torch.optim.SGD([w, b], lr=0.01)

losses = []

for epoch in range(100):
    # Forward
    y_pred = w * X + b
    loss = torch.mean((y_pred - y_true) ** 2)
    losses.append(loss.item())
    
    # The magic 3 lines:
    optimizer.zero_grad()   # 1. clear old gradients
    loss.backward()          # 2. compute new gradients
    optimizer.step()         # 3. update w and b
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}: loss={loss.item():.3f}, w={w.item():.3f}, b={b.item():.3f}")

print(f"\nFinal: w={w.item():.3f} (target: 3.0), b={b.item():.3f} (target: 7.0)")

### Why use an optimizer instead of manual updates?

1. **Cleaner code** — `optimizer.step()` updates ALL parameters at once
2. **Better optimizers** — SGD is basic. Adam (the most popular) adapts the learning rate per-parameter. We just swap one line:

```python
# optimizer = torch.optim.SGD([w, b], lr=0.01)   # basic
optimizer = torch.optim.Adam([w, b], lr=0.01)     # smarter — used in most LLMs
```

3. **Scales to any model** — whether you have 2 parameters or 175 billion, the 3-line loop is the same

Let's try Adam and compare:

In [ ]:
# Compare SGD vs Adam

def train_with_optimizer(opt_class, opt_name, lr=0.1):
    torch.manual_seed(42)
    X = torch.rand(50) * 10 - 5
    y_true = 3 * X + 7 + torch.randn(50) * 2
    
    w = torch.tensor(0.0, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    optimizer = opt_class([w, b], lr=lr)
    
    losses = []
    for epoch in range(100):
        y_pred = w * X + b
        loss = torch.mean((y_pred - y_true) ** 2)
        losses.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    return losses, w.item(), b.item()

sgd_losses, sgd_w, sgd_b = train_with_optimizer(torch.optim.SGD, "SGD", lr=0.01)
adam_losses, adam_w, adam_b = train_with_optimizer(torch.optim.Adam, "Adam", lr=0.1)

plt.figure(figsize=(10, 5))
plt.plot(sgd_losses, 'b-', label=f'SGD (final w={sgd_w:.2f}, b={sgd_b:.2f})', linewidth=2)
plt.plot(adam_losses, 'r-', label=f'Adam (final w={adam_w:.2f}, b={adam_b:.2f})', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('SGD vs Adam Optimizer')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Adam often converges faster because it adapts the learning rate for each parameter.")
print("Most LLM training uses Adam (or its variant AdamW).")

## 5. GPU / Apple Silicon Acceleration

Matrix math is "embarrassingly parallel" — thousands of multiplications can happen at the same time. GPUs have thousands of tiny cores built for exactly this.

On your Mac, PyTorch can use **MPS** (Metal Performance Shaders) — Apple's GPU framework.

In [ ]:
# Check what's available on your machine
print("Available devices:")
print(f"  CPU:  always available")
print(f"  CUDA: {torch.cuda.is_available()} (NVIDIA GPU)")
print(f"  MPS:  {torch.backends.mps.is_available()} (Apple Silicon GPU)")

# Pick the best available device
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"\nUsing device: {device}")

# Move tensors to GPU
a = torch.randn(1000, 1000).to(device)
b = torch.randn(1000, 1000).to(device)

# This multiplication runs on GPU!
c = a @ b
print(f"\nMultiplied two 1000×1000 matrices on {device}")
print(f"Result shape: {c.shape}, device: {c.device}")

In [ ]:
# Speed comparison: CPU vs GPU
import time

sizes = [500, 1000, 2000, 4000]
cpu_times = []
gpu_times = []

for size in sizes:
    # CPU
    a_cpu = torch.randn(size, size)
    b_cpu = torch.randn(size, size)
    start = time.time()
    for _ in range(5):
        _ = a_cpu @ b_cpu
    cpu_time = (time.time() - start) / 5
    cpu_times.append(cpu_time)
    
    # GPU (if available)
    if device.type != 'cpu':
        a_gpu = torch.randn(size, size, device=device)
        b_gpu = torch.randn(size, size, device=device)
        # Warmup
        _ = a_gpu @ b_gpu
        if device.type == 'mps':
            torch.mps.synchronize()
        start = time.time()
        for _ in range(5):
            _ = a_gpu @ b_gpu
        if device.type == 'mps':
            torch.mps.synchronize()
        gpu_time = (time.time() - start) / 5
        gpu_times.append(gpu_time)
    
    speedup = f", GPU: {gpu_time*1000:.1f}ms ({cpu_time/gpu_time:.1f}x faster)" if gpu_times else ""
    print(f"Size {size}×{size}: CPU: {cpu_time*1000:.1f}ms{speedup}")

if gpu_times:
    plt.figure(figsize=(8, 4))
    plt.plot(sizes, [t*1000 for t in cpu_times], 'bo-', label='CPU', linewidth=2)
    plt.plot(sizes, [t*1000 for t in gpu_times], 'ro-', label=f'{device.type.upper()}', linewidth=2)
    plt.xlabel('Matrix size')
    plt.ylabel('Time (ms)')
    plt.title('CPU vs GPU: Matrix Multiplication Speed')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    print("\nGPU gets relatively faster as matrices get bigger — perfect for LLMs!")
else:
    print("\nNo GPU available — CPU only. That's fine for learning!")

### Important rule: all tensors must be on the same device

```python
cpu_tensor = torch.randn(3)
gpu_tensor = torch.randn(3).to('mps')
result = cpu_tensor + gpu_tensor  # ERROR! Can't mix devices
```

Always move everything to the same device before doing math. The pattern:
```python
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
X = X.to(device)
model = model.to(device)
```

---

## 6. Mini Project: Nonlinear Function — Something Only PyTorch Can Do Easily

In Day 02, we fit a line. But what about a curve? The gradient formulas get nasty fast by hand. With PyTorch, we just write the forward pass and let autograd handle it.

In [ ]:
# Let's fit y = a*sin(b*x + c) + d  — 4 parameters!
# Try deriving gradients for this by hand... not fun.
# With PyTorch, we don't need to.

# Generate data from a sine curve with noise
torch.manual_seed(42)
X = torch.linspace(-5, 5, 100)
y_true = 2.5 * torch.sin(1.5 * X + 0.5) + 1.0 + torch.randn(100) * 0.3

# 4 learnable parameters — all start random
a = torch.tensor(1.0, requires_grad=True)
b_param = torch.tensor(1.0, requires_grad=True)
c = torch.tensor(0.0, requires_grad=True)
d = torch.tensor(0.0, requires_grad=True)

optimizer = torch.optim.Adam([a, b_param, c, d], lr=0.01)
losses = []

for epoch in range(2000):
    # Forward pass: our model
    y_pred = a * torch.sin(b_param * X + c) + d
    
    # Loss
    loss = torch.mean((y_pred - y_true) ** 2)
    losses.append(loss.item())
    
    # The magic 3 lines — works for ANY forward pass!
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f"Learned:  a={a.item():.2f}, b={b_param.item():.2f}, c={c.item():.2f}, d={d.item():.2f}")
print(f"Actual:   a=2.50, b=1.50, c=0.50, d=1.00")
print(f"Final loss: {losses[-1]:.4f}")

In [ ]:
# Visualize the fit + loss curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: data + learned curve
with torch.no_grad():  # no need to track gradients for plotting
    y_learned = a * torch.sin(b_param * X + c) + d

ax1.scatter(X.numpy(), y_true.numpy(), alpha=0.4, s=20, label='Noisy data')
ax1.plot(X.numpy(), y_learned.numpy(), 'r-', linewidth=2, label='Learned curve')
ax1.plot(X.numpy(), (2.5 * torch.sin(1.5 * X + 0.5) + 1.0).numpy(), 
         'g--', linewidth=2, alpha=0.5, label='True curve')
ax1.set_title('PyTorch Learned a Sine Curve!')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: loss curve
ax2.plot(losses, 'b-', linewidth=1)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Loss Curve')
ax2.set_yscale('log')  # log scale shows the improvement better
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("We NEVER wrote a single gradient formula.")
print("We just described the forward computation, and PyTorch did the rest.")
print("This is the power of autograd — it scales to ANY model, ANY complexity.")

### Why this matters:

In Day 02, we fit a line (`y = wx + b`) — 2 parameters, easy gradients by hand.

Here we fit a sine curve (`y = a*sin(b*x + c) + d`) — 4 parameters, gradients would be painful by hand. But the PyTorch code was **identical in structure**: forward pass → loss → `zero_grad()` → `backward()` → `step()`.

An LLM like GPT has **billions** of parameters with far more complex operations (matrix multiplies, softmax, layer norm, attention). The training code? Still the same 3 lines:

```python
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

---

## Exercises

1. **Break autograd on purpose:** Remove `w.grad.zero_()` from the manual training loop. What happens to the loss? Why?

2. **Try a different optimizer:** Replace `Adam` with `torch.optim.SGD` in the sine curve example. Does it still converge? Try different learning rates.

3. **Verify autograd:** For `y = 3x² + 2x + 1`, compute `dy/dx` both with `y.backward()` and by hand (answer: `6x + 2`). Do they match?

4. **GPU experiment:** If you have MPS available, time the sine curve training on CPU vs MPS. Is GPU faster for this small problem? (Hint: GPU has overhead — it's only faster for large matrix operations)

---

## Key Takeaways

| Concept | What It Does |
|---------|-------------|
| `torch.tensor(data)` | Creates a tensor (like NumPy array but with superpowers) |
| `requires_grad=True` | Tells PyTorch to track gradients for this tensor |
| `loss.backward()` | Computes all gradients automatically via chain rule |
| `tensor.grad` | Where the computed gradient is stored |
| `optimizer.zero_grad()` | Clears old gradients (must do before each backward!) |
| `optimizer.step()` | Updates all parameters using their gradients |
| `torch.no_grad()` | Disables gradient tracking (for inference / evaluation) |
| `tensor.to(device)` | Moves tensor to GPU/CPU |

### The evolution across 3 days:

```
Day 01: "Here's how matrix math works"          (NumPy)
Day 02: "Here's how to compute gradients"        (by hand)
Day 03: "Let PyTorch compute gradients for you"   (autograd)
```

**Tomorrow:** We'll handle real data — loading text files, splitting into batches, and tokenization (turning words into numbers that a model can process).